In [1]:
import os
import time
from pyspark.sql import SparkSession

In [2]:
MINIO_AK = os.environ.get('MINIO_ACCESS_KEY')
MINIO_SK = os.environ.get('MINIO_SECRET_KEY')
MASTER_URL = os.environ.get('SPARK_MASTER', 'local[*]')

In [3]:
import os
from pyspark.sql import SparkSession

# =====================================================================
# 0. LIMPANDO SESSÕES ZUMBIS (O Segredo do Jupyter!)
# =====================================================================
try:
    spark.stop()
    print("Sessão anterior encerrada com sucesso.")
except:
    pass

# =====================================================================
# 1. CONFIGURAÇÃO DA SESSÃO SPARK COM SUPORTE A S3/MINIO
# =====================================================================
MINIO_AK = os.environ.get('MINIO_ACCESS_KEY', 'arenalake')
MINIO_SK = os.environ.get('MINIO_SECRET_KEY', 'arenalake123')

print("🚀 Inicializando o Motor do Apache Spark (Nova Sessão Limpa)...")

spark = SparkSession.builder \
    .appName("Job_Jupyter_ArenaLake") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_AK) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SK) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✅ Conectado ao Cluster com sucesso! Atualize a aba Spark Process.")

🚀 Inicializando o Motor do Apache Spark (Nova Sessão Limpa)...
:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/coder/.ivy2/cache
The jars for the packages stored in: /home/coder/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db4593c8-8ab6-4ec0-9ddf-77e83125d941;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 556ms :: artifacts dl 56ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

✅ Conectado ao Cluster com sucesso! Atualize a aba Spark Process.


In [4]:
caminho_origem = "s3a://bronze/MOCK_DATA.csv"
print(f"📖 Lendo o arquivo CSV de origem: {caminho_origem}")

df = spark.read.csv(caminho_origem, header=True, inferSchema=True)
df.show(5)

📖 Lendo o arquivo CSV de origem: s3a://bronze/MOCK_DATA.csv


+---+----------+-----------+--------------------+------+--------------+
| id|first_name|  last_name|               email|gender|    ip_address|
+---+----------+-----------+--------------------+------+--------------+
|  1|     Shaun|   Parradye|sparradye0@skype.com|  Male|77.211.251.134|
|  2|       Ade|  Blaxlande|ablaxlande1@uiuc.edu|  Male|  18.72.169.16|
|  3|  Rochette|Ellesworthe|rellesworthe2@inf...|Female|   70.55.63.91|
|  4|    Teador|  LaBastida|tlabastida3@istoc...|  Male| 92.211.204.60|
|  5|     Elroy|   Cotgrove|  ecotgrove4@ucoz.ru|  Male|  8.81.191.147|
+---+----------+-----------+--------------------+------+--------------+
only showing top 5 rows



In [5]:
caminho_destino = "s3a://bronze/tabelas/MOCK_DATA_PARQUET"
print(f"💾 Convertendo e salvando como Tabela Parquet em: {caminho_destino}")

# Salva particionado (se já existir, ele sobrescreve)
df.write.mode("overwrite").parquet(caminho_destino)
print("✅ Tabela salva com sucesso!")

💾 Convertendo e salvando como Tabela Parquet em: s3a://bronze/tabelas/MOCK_DATA_PARQUET


✅ Tabela salva com sucesso!


In [6]:
print(f"🔄 Lendo a nova Tabela Parquet para confirmar...")
df_novo = spark.read.parquet(caminho_destino)

# Vamos fazer uma contagem rápida por Gênero para gerar um "Job" visível pro cluster calcular
print("📊 Agrupando por Gênero:")
df_novo.groupBy("gender").count().show()

🔄 Lendo a nova Tabela Parquet para confirmar...
📊 Agrupando por Gênero:


+-----------+-----+
|     gender|count|
+-----------+-----+
|Genderqueer|   12|
|    Agender|   17|
|     Female|  449|
| Polygender|   23|
|   Bigender|   14|
| Non-binary|   21|
|       Male|  441|
|Genderfluid|   23|
+-----------+-----+



In [ ]:
print("\n🔥 SUCESSO ABSOLUTO! 🔥")
print("👉 Vá para o portal ArenaLake na aba '⚡ Spark Process'.")
print("O Job vai ficar ativo por 60 segundos para você conseguir vê-lo no painel de Running Applications...")

for i in range(60, 0, -1):
    print(f"Encerrando sessão em {i} segundos...", end="\r")
    time.sleep(1)

spark.stop()
print("\nSessão encerrada.")

In [ ]:
# Lendo a Tabela Parquet completa (o Spark entende que a pasta inteira é uma tabela só)
caminho_tabela = "s3a://bronze/tabelas/MOCK_DATA_PARQUET"

print("🔍 Puxando os dados da tabela Parquet do Data Lake...\n")
df_parquet = spark.read.parquet(caminho_tabela)

# Mostra as 10 primeiras linhas para provar que a conversão foi perfeita
df_parquet.show(10)

# Mostra o Schema (Você vai ver que o Spark preservou os tipos de dados nativos)
df_parquet.printSchema()

In [ ]:
# Define o caminho da tabela
caminho_destino = "s3a://bronze/tabelas/MOCK_DATA_PARQUET"

# Recupera o FileSystem que o Spark está usando
sc = spark.sparkContext
path_obj = sc._gateway.jvm.org.apache.hadoop.fs.Path(caminho_destino)
fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

# Deleta a pasta e tudo dentro dela recursivamente
if fs.exists(path_obj):
    fs.delete(path_obj, True)
    print(f"🗑️ Tabela em {caminho_destino} deletada com sucesso!")
else:
    print("⚠️ O caminho especificado não existe.")
